In [0]:
from pyspark.sql.functions import col, rand, floor, lit, concat, explode, array
import pyspark.sql.functions as F

# ==========================================
# 1. AQE (Adaptive Query Execution)
# ==========================================
print("✅ Mechanizm AQE jest natywnie zarządzany przez platformę Databricks Serverless!")

# ==========================================
# 2. Przygotowanie danych do demonstracji (Odczyt z Silver)
# ==========================================
payroll_df = spark.read.table("dbw_showcase.default.payroll_silver")

# Tworzymy małą, sztuczną tabelę wymiarów (Dimension) - budżety biur
offices_data = [
    ("Krakow HQ", 1000000), ("Wroclaw", 500000), 
    ("Warsaw", 750000), ("Gdansk", 400000), ("Poznan", 300000)
]
offices_dim = spark.createDataFrame(offices_data, ["office_location", "budget"])

# ==========================================
# 3. SALTING - Inżynieria rozpraszania kluczy (Data Skew Mitigation)
# ==========================================
print("⏳ Przygotowuję Salting dla symulowanego Data Skew (Krakow HQ)...")

SALT_BINS = 5 # Rozbijamy przeciążony klucz na 5 mniejszych kubełków

# Krok A: Solenie dużej tabeli faktów
payroll_salted = payroll_df.withColumn(
    "salted_office", 
    concat(col("office_location"), lit("_"), floor(rand() * SALT_BINS))
)

# Krok B: Replikacja małej tabeli wymiarów przy użyciu funkcji explode
salt_array = array(*[lit(i) for i in range(SALT_BINS)])
offices_salted = offices_dim.withColumn("salt", explode(salt_array)) \
                          .withColumn("salted_office", concat(col("office_location"), lit("_"), col("salt")))

# ==========================================
# Krok C: Optymalny Skew Join z aliasami
# ==========================================
optimized_join_df = payroll_salted.alias("fact").join(
    offices_salted.alias("dim"),
    "salted_office",
    "inner"
)

# 4. Agregacja i wyświetlenie wyników z uwzględnieniem oryginalnego klucza
print("📊 Wynik złączenia z uwzględnieniem Saltingu:")
display(
    optimized_join_df.groupBy("fact.office_location").agg(
        F.count("*").alias("transaction_count"),
        F.first("budget").alias("office_budget")
    ).orderBy(F.desc("transaction_count"))
)